In [ ]:
import os
from pathlib import Path

from datasets import load_dataset
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
import pandas as pd

# Dataset to download from Hugging Face
DATASET_REPO_ID = "ronantakizawa/japanese-honorifics"
DATASET_DIR_NAME = "japanese-honorifics"

DATA_DIR = "data/"

def download_honorifics_dataset(data_dir: str = str(DATA_DIR),
                                repo_id: str = DATASET_REPO_ID,
                                token: str | None = None,
                                overwrite: bool = False) -> Path:
    """Downloads the Japanese honorifics dataset from Hugging Face.

    Downloads the dataset via the `datasets` library and stores it on disk
    in the Hugging Face `datasets` format (Arrow), which can be reloaded
    later with `datasets.load_from_disk()`.

    Args:
        data_dir: Directory to save the dataset into. Defaults to the
            project-level `data/` directory.
        repo_id: The Hugging Face dataset repository to download.
        token: Optional Hugging Face access token, required only for
            gated/private datasets.
        overwrite: Whether to re-download the dataset even if a local
            copy already exists.

    Returns:
        The Path to the saved dataset directory.
    """
    data_dir_path = Path(data_dir)
    save_dir = data_dir_path / DATASET_DIR_NAME

    # Skip downloading if the dataset already exists locally
    if save_dir.exists() and not overwrite:
        print(f"[INFO] Dataset already exists at {save_dir}, skipping download.")
        return save_dir
    

    
    print(f"[INFO] Downloading dataset {repo_id} from Hugging Face...")
    dataset = load_dataset(repo_id, token=token)
    
    # dataset_kenjogo = dataset["train"]["kenjogo"]
    # dataset_sonkeigo = dataset["train"]["sonkeigo"]
    # dataset_teineigo = dataset["train"]["teineigo"]
    # dataset_base_sentence = dataset["train"]["base_sentence"]
    
    df_default = {
        "sentence": [],
        "sentence_politeness": []
    }
    
    dataset_data = dataset["train"]
    
    
    
    for row in dataset_data:
        df_default["sentence"].append(row["base_sentence"])
        df_default["sentence_politeness"].append("base_sentence")
        df_default["sentence"].append(row["teineigo"])
        df_default["sentence_politeness"].append("teineigo")
        df_default["sentence"].append(row["sonkeigo"])
        df_default["sentence_politeness"].append("sonkeigo")
        df_default["sentence"].append(row["kenjogo"])
        df_default["sentence_politeness"].append("kenjogo")
    
    df_final_default = pd.DataFrame(df_default)
    print(df_final_default)
    
    
    
    
    return save_dir

def main():
    """Downloads the dataset, using HF_TOKEN from the environment if set.

    Best practice for secrets: the token is read from an environment
    variable (loaded from `.env` via python-dotenv), never hardcoded.
    """
    # Load HF_TOKEN from the project-level .env file (if present)
    load_dotenv()
    token = os.getenv("HF_TOKEN") or None

    save_dir = download_honorifics_dataset(data_dir=str(DATA_DIR), token=token)
    print(f"[INFO] Dataset ready at: {save_dir}")

main()

[INFO] Downloading dataset ronantakizawa/japanese-honorifics from Hugging Face...
             sentence sentence_politeness
0             食事を食べる。       base_sentence
1           ご食事を食べます。            teineigo
2        ご食事を召し上がります。            sonkeigo
3         ご食事をいただきます。             kenjogo
4               本を読む。       base_sentence
..                ...                 ...
455  新しい車を買わせていただきます。             kenjogo
456           そろそろ帰る。       base_sentence
457         そろそろ帰ります。            teineigo
458     そろそろお帰りになります。            sonkeigo
459    そろそろおいとまいたします。             kenjogo

[460 rows x 2 columns]
[INFO] Dataset ready at: data/japanese-honorifics
